In [1]:
import os, re, unicodedata, random, time, itertools, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp

D = r"C:\ProjectHub\neurahack\drive-download-20261127T064654Z-1-001"          # <- folder with the 3 csvs
tr, va, te = [pd.read_csv(os.path.join(D, f)) for f in ("train.csv", "validation.csv", "test.csv")]
print(tr.columns.tolist()); print(len(tr), len(va), len(te))

# ---- text normalisation (compact version of Shanto's normalize.robust) ----
HG = str.maketrans({"а":"a","е":"e","о":"o","р":"p","с":"c","у":"y","х":"x","і":"i","ѕ":"s","ԁ":"d","һ":"h",
                    "А":"A","В":"B","Е":"E","К":"K","М":"M","Н":"H","О":"O","Р":"P","С":"C","Т":"T","Х":"X"})
def robust(t):
    t = unicodedata.normalize("NFKC", str(t))
    t = "".join(c for c in t if unicodedata.category(c) != "Cf").translate(HG)
    return " ".join(re.sub(r"(.)\1{2,}", r"\1\1", t).split())

# ---- pool = train + validation; test stays locked ----
pool = pd.concat([tr, va], ignore_index=True)
if "is_synthetic" not in pool: pool["is_synthetic"] = False
pool["is_synthetic"] = pool.is_synthetic.astype(str).str.lower().isin(["true", "1"])
pool["text_n"] = pool.text.map(robust)
# group = same message modulo casing/punctuation (variants of one source stay together)
pool["grp"] = pool.text.map(lambda s: re.sub(r"[\W_]+", "", unicodedata.normalize("NFKC", str(s)).lower()))
y = pool.label.to_numpy(); G = pool.grp.factorize()[0]; ORIG = ~pool.is_synthetic.to_numpy()
print("rows", len(pool), "| originals", ORIG.sum(), "| groups", len(set(G)), "| class counts", pd.Series(y[ORIG]).value_counts().to_dict())

# ---- noisy copy of every text, to estimate the hidden robustness slice ----
def corrupt(t, rng, p=0.08):
    out = []
    for ch in str(t):
        r = rng.random()
        if r < p/4:   out.append({"a":"а","e":"е","o":"о","p":"р","c":"с"}.get(ch, ch))   # Cyrillic lookalike
        elif r < p/2: out.append(ch + "\u200b")                                            # zero-width char
        elif r < 3*p/4: out.append(ch * 3)                                                 # repeat
        elif r < p:   continue                                                             # delete
        else:         out.append(ch)
    return "".join(out)
rng = random.Random(0)
pool["text_noisy_n"] = [robust(corrupt(t, rng)) for t in pool.text]

['id', 'text', 'label', 'techniques', 'action', 'source_id', 'is_synthetic']
569 17 17
rows 586 | originals 69 | groups 69 | class counts {0: 26, 1: 10, 5: 9, 3: 9, 2: 8, 4: 7}


In [2]:
import torch
from sentence_transformers import SentenceTransformer
os.makedirs("emb_cache", exist_ok=True)

def head_tail(t, n):                       # keep start + end: the scam ask often comes in the LAST turns
    return t if len(t) <= n else t[: n // 4] + " ... " + t[-(3 * n // 4):]

ENCODERS = {  # key: (hf name, prefix)
 "e5s":  ("intfloat/multilingual-e5-small", "query: "),
 "e5b":  ("intfloat/multilingual-e5-base",  "query: "),
 "e5l":  ("intfloat/multilingual-e5-large", "query: "),
 "bgem3":("BAAI/bge-m3", ""),
 "mini": ("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", ""),
 "mpnet":("sentence-transformers/paraphrase-multilingual-mpnet-base-v2", ""),
 "labse":("sentence-transformers/LaBSE", ""),
}
def embed(key, texts, tag, max_chars=1200, max_len=256):
    f = f"emb_cache/{key}_{tag}_{max_chars}_{max_len}.npy"
    if os.path.exists(f): return np.load(f)
    name, prefix = ENCODERS[key]
    m = SentenceTransformer(name, device="cuda"); m.max_seq_length = max_len; m.half()
    v = m.encode([prefix + head_tail(t, max_chars) for t in texts], batch_size=64,
                 normalize_embeddings=True, show_progress_bar=False)
    np.save(f, v); del m; torch.cuda.empty_cache(); return v

EMB = {}
for k in ENCODERS:
    try:
        t0 = time.time()
        EMB[k] = (embed(k, pool.text_n, "clean"), embed(k, pool.text_noisy_n, "noisy"))
        print(k, EMB[k][0].shape, round(time.time() - t0, 1), "s")
    except Exception as e:
        print("SKIP", k, str(e)[:100])

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

... (94 outputs left out)

2_Dense/model.safetensors: reconstructing file:   0%|          |  0.00B / 2.36MB            

2_Dense/model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

labse (586, 768) 76.2 s


In [3]:
def feats(spec, tr_i, ev_i, noisy):
    """spec -> (X_train, X_eval). TF-IDF is fit on the training rows of the fold only."""
    kind = spec[0]
    if kind == "emb":
        E = EMB[spec[1]][1 if noisy else 0]; Etr = EMB[spec[1]][0]
        return Etr[tr_i], E[ev_i]
    txt_tr = pool.text_n.iloc[tr_i]; txt_ev = (pool.text_noisy_n if noisy else pool.text_n).iloc[ev_i]
    if kind == "char":
        v = TfidfVectorizer(analyzer="char_wb", ngram_range=spec[1], min_df=1, sublinear_tf=True)
    elif kind == "word":
        v = TfidfVectorizer(ngram_range=spec[1], min_df=1, sublinear_tf=True)
    elif kind == "stack":                                    # ("stack", [subspecs], [weights])
        parts = [feats(s, tr_i, ev_i, noisy) for s in spec[1]]
        w = spec[2]
        return (sp.hstack([sp.csr_matrix(p[0]) * wi for p, wi in zip(parts, w)]).tocsr(),
                sp.hstack([sp.csr_matrix(p[1]) * wi for p, wi in zip(parts, w)]).tocsr())
    return v.fit_transform(txt_tr), v.transform(txt_ev)

CLF = {
 "lr_C0.3": lambda: LogisticRegression(C=0.3, max_iter=3000, class_weight="balanced"),
 "lr_C1":   lambda: LogisticRegression(C=1,   max_iter=3000, class_weight="balanced"),
 "lr_C3":   lambda: LogisticRegression(C=3,   max_iter=3000, class_weight="balanced"),
 "lr_C10":  lambda: LogisticRegression(C=10,  max_iter=3000, class_weight="balanced"),
 "lr_C30":  lambda: LogisticRegression(C=30,  max_iter=3000, class_weight="balanced"),
 "ridge1":  lambda: RidgeClassifier(alpha=1.0, class_weight="balanced"),
 "svc_C1":  lambda: LinearSVC(C=1, class_weight="balanced"),
 "knn5":    lambda: KNeighborsClassifier(5, metric="cosine", weights="distance"),
}

def run(spec_name, spec, clf_name, train_mode, seeds=(0, 1, 2, 3, 4)):
    clean, noisy = [], []
    for s in seeds:
        for tr_i, va_i in StratifiedGroupKFold(5, shuffle=True, random_state=s).split(pool, y, G):
            if train_mode == "orig": tr_i = tr_i[ORIG[tr_i]]
            ev_i = va_i[ORIG[va_i]]                                   # score on ORIGINAL messages only
            if len(set(y[tr_i])) < 2 or len(ev_i) == 0: continue
            try:
                Xtr, Xc = feats(spec, tr_i, ev_i, False); _, Xn = feats(spec, tr_i, ev_i, True)
                m = CLF[clf_name]().fit(Xtr, y[tr_i])
                clean.append(f1_score(y[ev_i], m.predict(Xc), average="macro"))
                noisy.append(f1_score(y[ev_i], m.predict(Xn), average="macro"))
            except Exception as e:
                return None
    return dict(spec=spec_name, clf=clf_name, train=train_mode,
                f1=np.mean(clean), sd=np.std(clean), f1_noisy=np.mean(noisy), n=len(clean))

In [4]:
SPECS = {
 "char2-5":      ("char", (2, 5)),
 "char1-4":      ("char", (1, 4)),
 "word1-2":      ("word", (1, 2)),
 "char+word":    ("stack", [("char", (2, 5)), ("word", (1, 2))], [1, 1]),
}
for k in EMB:
    SPECS[f"emb_{k}"] = ("emb", k)
    SPECS[f"emb_{k}+char"] = ("stack", [("emb", k), ("char", (2, 5))], [1.0, 0.5])
    SPECS[f"emb_{k}+char_w1"] = ("stack", [("emb", k), ("char", (2, 5))], [1.0, 1.0])
if "e5b" in EMB and "bgem3" in EMB:
    SPECS["e5b+bgem3"] = ("stack", [("emb", "e5b"), ("emb", "bgem3")], [1, 1])
    SPECS["e5l+bgem3+char"] = ("stack", [("emb", "e5l"), ("emb", "bgem3"), ("char", (2, 5))], [1, 1, 0.5])

results, t0 = [], time.time()
for (sn, spec), cn, tm in itertools.product(SPECS.items(), CLF, ("orig", "all")):
    if sn.startswith(("char", "word")) is False and "emb" not in sn: continue
    r = run(sn, spec, cn, tm)
    if r: results.append(r)
print(len(results), "configs in", round((time.time() - t0) / 60, 1), "min")

R = pd.DataFrame(results)
R["score"] = 0.7 * R.f1 + 0.3 * R.f1_noisy        # one number: mostly clean, partly robust
R = R.sort_values("score", ascending=False)
R.to_csv("benchmark_results.csv", index=False)
print(R.head(20).round(3).to_string(index=False))

176 configs in 3.5 min
     spec     clf train    f1    sd  f1_noisy  n  score
emb_bgem3  ridge1  orig 0.806 0.117     0.787 25  0.800
emb_bgem3  svc_C1  orig 0.807 0.121     0.781 25  0.799
emb_bgem3   lr_C3  orig 0.788 0.107     0.778 25  0.785
emb_bgem3  lr_C10  orig 0.787 0.122     0.778 25  0.784
emb_bgem3  lr_C30  orig 0.789 0.114     0.765 25  0.782
emb_bgem3  svc_C1   all 0.780 0.118     0.767 25  0.776
emb_bgem3   lr_C1  orig 0.769 0.125     0.771 25  0.770
emb_bgem3  ridge1   all 0.767 0.113     0.768 25  0.767
emb_bgem3 lr_C0.3  orig 0.751 0.119     0.747 25  0.750
emb_bgem3  lr_C30   all 0.743 0.118     0.751 25  0.745
  emb_e5l  ridge1   all 0.752 0.127     0.728 25  0.745
emb_bgem3  lr_C10   all 0.741 0.112     0.751 25  0.744
  emb_e5l  lr_C10  orig 0.751 0.132     0.688 25  0.732
  emb_e5l  svc_C1   all 0.735 0.125     0.710 25  0.727
emb_bgem3   lr_C3   all 0.712 0.116     0.733 25  0.718
emb_bgem3 lr_C0.3   all 0.709 0.109     0.715 25  0.711
emb_bgem3   lr_C1   all 0

In [5]:
import os
print(os.getcwd())
print(sorted(os.listdir("emb_cache")))
print(os.path.exists("benchmark_results.csv"))

C:\Users\onimr
['bgem3_clean_1200_256.npy', 'bgem3_noisy_1200_256.npy', 'e5b_clean_1200_256.npy', 'e5b_noisy_1200_256.npy', 'e5l_clean_1200_256.npy', 'e5l_noisy_1200_256.npy', 'e5s_clean_1200_256.npy', 'e5s_noisy_1200_256.npy', 'labse_clean_1200_256.npy', 'labse_noisy_1200_256.npy', 'mini_clean_1200_256.npy', 'mini_noisy_1200_256.npy', 'mpnet_clean_1200_256.npy', 'mpnet_noisy_1200_256.npy']
True


In [6]:
import re
src = open(r"C:\ProjectHub\neurahack\synth_data.py", encoding="utf-8").read()
print(len(src), "chars")
print(re.findall(r"^def \w+\(.*\):", src, flags=re.M)[:40])
print(src[:1500])          # the docstring at the top usually explains usage

78875 chars
['def _amt(rng, lo, hi, step=50):', 'def _fill(slot, rng):', 'def _render(tpl, rng):', 'def _compose_benign(rng):', 'def _compose(rng):', 'def _lang(text):', 'def _split_sentences(text):', 'def _as_conversation(text, rng, scam):', 'def _as_group_chat(text, rng):', 'def _as_email(text, rng, org):', 'def _as_group_long(text, rng):', 'def _as_call(text, rng):', 'def _warning_about(scam_text, rng):', 'def _techs_str(ts):', 'def _code_mix(text, rng, p=0.6):', 'def generate(n=4000, seed=7, mix=None, compose=0.55):']
"""MEMBER A. Synthetic labelled training messages, generated by code (seeded, reproducible, declared in README).

The organisers' Data tab allows synthetic data. Nothing here is derived from test.csv: every template was written by
hand from the official labelling guide (classes, priority order, technique/action conventions).

Labels follow the guide:
  label   0 legit, 1 MFS/OTP/credential, 2 prize, 3 fake job/investment, 4 parcel phishing, 5 promo spam
  T1 organisat

In [8]:
import sys, importlib, inspect
sys.path.insert(0, r"C:\ProjectHub\neurahack")
import synth_data; importlib.reload(synth_data)

print(inspect.getsource(synth_data.generate)[:2500])      # shows what mix= and compose= do

def generate(n=4000, seed=7, mix=None, compose=0.55):
    """n labelled synthetic rows. mix = class weights (default leans legit, like a real inbox)."""
    rng = random.Random(seed)
    mix = mix or {0: 0.28, 1: 0.15, 2: 0.13, 3: 0.13, 4: 0.13, 5: 0.18}
    classes, weights = zip(*mix.items())
    out = []
    while len(out) < n:
        c = rng.choices(classes, weights)[0]
        if c in T and rng.random() < compose:
            text, lab_, techs_, act_, tid = _compose(rng)
            r = rng.random()
            if r < 0.2:
                text = _as_conversation(text, rng, scam=True)
            elif r < 0.27:
                text = _as_call(text, rng)
            elif r < 0.40:
                text = _as_email(text, rng, None)
            out.append((text, lab_, techs_, act_, tid))
            continue
        if c not in T and rng.random() < compose * 0.6:
            out.append(_compose_benign(rng))
            continue
        if c in T:
            ti = rng.randrange(len(T[c

In [9]:
syn_df = synth_data.generate(n=3000, seed=7)
print(syn_df.shape, syn_df.columns.tolist())
print(syn_df.label.value_counts().sort_index().to_dict())
print(syn_df.sample(8, random_state=0)[["text", "label"]].to_string(max_colwidth=160))

(2861, 5) ['text', 'label', 'techniques', 'action', 'tpl']
{0: 748, 1: 513, 2: 353, 3: 361, 4: 331, 5: 555}
                                                                                                                                                                 text  label
1567                                                      Hasan: আপনি 18450 টাকার রিওয়ার্ড পয়েন্ট পেয়েছেন। ||| You: ki bolen? ||| Hasan: আপনার নাম ও ঠিকানা পাঠান।      2
547                                                                             Boss: Meeting moved to 8 PM. Please join the Google Meet link in the calendar invite.      0
33                                                                                                                                                  কাল কি অফিস খোলা?      0
2575  Caller: Barir sobai valo? ||| You: ji sobai valo ||| Caller: Barir sobai valo? ||| You: ji sobai valo ||| Caller: Ami Mymensingh theke bolchi ||| You: accha...      0
852   [You] Kemon achen? ||

In [10]:
import re
def skeleton(t):                                  # strip digits and punctuation, lowercase
    return re.sub(r"[\W\d_]+", " ", str(t).lower()).strip()
sk = syn_df.text.map(skeleton)
print("rows:", len(syn_df), "| unique texts:", syn_df.text.nunique(), "| unique skeletons:", sk.nunique())

# first 6 words as a crude template id
head = sk.map(lambda s: " ".join(s.split()[:6]))
print("distinct 6-word openings:", head.nunique())
print(head.value_counts().head(8))

# language mix
print(syn_df.text.map(synth_data._lang).value_counts().to_dict())

# overlap with your real messages (must be 0)
real = set(pool.grp)
g = syn_df.text.map(lambda s: re.sub(r"[\W_]+", "", unicodedata.normalize("NFKC", str(s)).lower()))
print("overlap with real messages:", g.isin(real).sum())

rows: 2861 | unique texts: 2861 | unique skeletons: 2758
distinct 6-word openings: 2028
text
সতর ক থ ক ন এই                                 21
call transcript caller sir apnar ektu          16
subject re your recent activity dear           15
call transcript caller assalamu alaikum you    14
call transcript caller hello sir do            14
bhai dekh ki scam message ashche               13
call transcript caller bhai ki obostha         12
call transcript caller hello apu ektu          12
Name: count, dtype: int64
{'en': 1631, 'bn': 709, 'bl': 521}
overlap with real messages: 4


In [12]:
real = set(pool.grp)
syn_grp = syn_df.text.map(lambda s: re.sub(r"[\W_]+", "", str(s).lower()))
print("overlap with real messages:", syn_grp.isin(real).sum())

overlap with real messages: 5


In [13]:
def embed_new(key, texts, max_chars=1200, max_len=256):
    name, prefix = ENCODERS[key]
    m = SentenceTransformer(name, device="cuda"); m.max_seq_length = max_len; m.half()
    v = m.encode([prefix + head_tail(t, max_chars) for t in texts], batch_size=64,
                 normalize_embeddings=True, show_progress_bar=False)
    del m; torch.cuda.empty_cache(); return v

In [14]:
syn_df = synth_data.generate(n=6000, seed=7)
syn_df["text_n"] = syn_df.text.map(robust)
t0 = time.time()
Es_all = embed_new("bgem3", syn_df.text_n.tolist())
print("encoded", len(syn_df), "in", round(time.time() - t0), "s")   # use this to estimate 300k

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

encoded 5638 in 18 s


In [15]:
def run_synth2(n_syn, w_syn, clf_name="lr_C3", enc="bgem3", seeds=range(5)):
    idx = np.random.RandomState(0).permutation(len(syn_df))[:n_syn]
    Es, ys = Es_all[idx], syn_df.label.to_numpy()[idx]
    out = []
    for s in seeds:
        for tr_i, va_i in StratifiedGroupKFold(5, shuffle=True, random_state=s).split(pool, y, G):
            tr_i = tr_i[ORIG[tr_i]]; ev_i = va_i[ORIG[va_i]]
            Xtr = np.vstack([EMB[enc][0][tr_i], Es]) if n_syn else EMB[enc][0][tr_i]
            ytr = np.r_[y[tr_i], ys] if n_syn else y[tr_i]
            sw = np.r_[np.ones(len(tr_i)), np.full(n_syn, w_syn)] if n_syn else None
            m = CLF[clf_name]().fit(Xtr, ytr, sample_weight=sw)
            out.append(f1_score(y[ev_i], m.predict(EMB[enc][0][ev_i]), average="macro"))
    return round(np.mean(out), 3), round(np.std(out), 3)

for n in (0, 300, 1000, 3000, 6000):
    for w in ((1.0,) if n == 0 else (1.0, 0.2, 0.05)):
        print(n, w, run_synth2(n, w))

0 1.0 (np.float64(0.788), np.float64(0.107))
300 1.0 (np.float64(0.964), np.float64(0.045))
300 0.2 (np.float64(0.938), np.float64(0.065))
300 0.05 (np.float64(0.856), np.float64(0.107))
1000 1.0 (np.float64(0.965), np.float64(0.047))
1000 0.2 (np.float64(0.953), np.float64(0.05))
1000 0.05 (np.float64(0.912), np.float64(0.069))
3000 1.0 (np.float64(0.969), np.float64(0.049))
3000 0.2 (np.float64(0.965), np.float64(0.047))
3000 0.05 (np.float64(0.953), np.float64(0.05))


ValueError: sample_weight.shape == (6058,), expected (5696,)!

In [16]:
def run_synth3(n_syn, w_syn, tau=None, clf_name="lr_C3", enc="bgem3", seeds=range(5)):
    n_syn = min(n_syn, len(syn_df))
    idx = np.random.RandomState(0).permutation(len(syn_df))[:n_syn]
    Es, ys = Es_all[idx], syn_df.label.to_numpy()[idx]
    out, kept = [], []
    for s in seeds:
        for tr_i, va_i in StratifiedGroupKFold(5, shuffle=True, random_state=s).split(pool, y, G):
            tr_i = tr_i[ORIG[tr_i]]; ev_i = va_i[ORIG[va_i]]
            keep = np.ones(len(Es), bool)
            if tau is not None:                                   # drop synthetic rows too close to ANY held-out message
                keep = (Es @ EMB[enc][0][va_i].T).max(1) < tau
            kept.append(keep.mean())
            Xtr = np.vstack([EMB[enc][0][tr_i], Es[keep]])
            ytr = np.r_[y[tr_i], ys[keep]]
            sw  = np.r_[np.ones(len(tr_i)), np.full(keep.sum(), w_syn)]
            m = CLF[clf_name]().fit(Xtr, ytr, sample_weight=sw)
            out.append(f1_score(y[ev_i], m.predict(EMB[enc][0][ev_i]), average="macro"))
    return round(np.mean(out), 3), round(np.std(out), 3), f"kept {np.mean(kept):.0%}"

print("no synth     ", run_synth3(0, 1.0))
for tau in (None, 0.90, 0.85, 0.80, 0.75):
    print("tau", tau, run_synth3(3000, 1.0, tau))

C:\Users\onimr\AppData\Local\Temp\ipykernel_3820\3064925336.py:12: RuntimeWarning: Mean of empty slice
  kept.append(keep.mean())
C:\Users\onimr\colab-env\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
C:\Users\onimr\AppData\Local\Temp\ipykernel_3820\3064925336.py:12: RuntimeWarning: Mean of empty slice
  kept.append(keep.mean())
C:\Users\onimr\colab-env\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
C:\Users\onimr\AppData\Local\Temp\ipykernel_3820\3064925336.py:12: RuntimeWarning: Mean of empty slice
  kept.append(keep.mean())
C:\Users\onimr\colab-env\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
C:\Users\onimr\AppData\Local\Temp\ipykernel_3820\3064925336.py:12: RuntimeWarning: Mean of empty slice
  k

no synth      (np.float64(0.788), np.float64(0.107), 'kept nan%')
tau None (np.float64(0.969), np.float64(0.049), 'kept 100%')
tau 0.9 (np.float64(0.968), np.float64(0.052), 'kept 100%')
tau 0.85 (np.float64(0.968), np.float64(0.052), 'kept 99%')
tau 0.8 (np.float64(0.964), np.float64(0.051), 'kept 99%')
tau 0.75 (np.float64(0.958), np.float64(0.056), 'kept 98%')


In [17]:
Es, ys = Es_all, syn_df.label.to_numpy()
for cn in ("lr_C3", "lr_C10", "ridge1"):
    m = CLF[cn]().fit(Es, ys)
    pred = m.predict(EMB["bgem3"][0][ORIG])
    print(cn, "synthetic-only -> real macro-F1:", round(f1_score(y[ORIG], pred, average="macro"), 3))
from sklearn.metrics import classification_report
print(classification_report(y[ORIG], pred))

lr_C3 synthetic-only -> real macro-F1: 0.981
lr_C10 synthetic-only -> real macro-F1: 1.0
ridge1 synthetic-only -> real macro-F1: 0.981
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        26
           1       1.00      0.90      0.95        10
           2       0.89      1.00      0.94         8
           3       1.00      1.00      1.00         9
           4       1.00      1.00      1.00         7
           5       1.00      1.00      1.00         9

    accuracy                           0.99        69
   macro avg       0.98      0.98      0.98        69
weighted avg       0.99      0.99      0.99        69



In [18]:
for tau in (0.70, 0.65, 0.60):
    print("tau", tau, run_synth3(3000, 1.0, tau))

tau 0.7 (np.float64(0.937), np.float64(0.067), 'kept 96%')
tau 0.65 (np.float64(0.937), np.float64(0.067), 'kept 85%')
tau 0.6 (np.float64(0.933), np.float64(0.068), 'kept 57%')


##**Cell 1: build the training set**



In [19]:
import re, time, unicodedata, numpy as np, pandas as pd, torch
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression

COMP = r"C:\ProjectHub\neurahack\nsucec-neurahack-2026-preliminary"
real = pd.read_csv(os.path.join(COMP, "sample.csv"))          # the 86 official labeled rows
real["text_n"] = real.text.map(robust)

TECH = [f"T{i}" for i in range(1, 7)]
def tech_matrix(s):
    return np.array([[int(t in str(v).split("|")) for t in TECH] for v in s])

# drop synthetic rows that are exact copies of real messages
grp = lambda s: re.sub(r"[\W_]+", "", unicodedata.normalize("NFKC", str(s)).lower())
real_grp = set(real.text.map(grp))
keep = ~syn_df.text.map(grp).isin(real_grp).to_numpy()
syn = syn_df[keep].reset_index(drop=True); Es = Es_all[keep]
print("synthetic rows kept:", len(syn), "of", len(syn_df))

# encoder, loaded once (same settings as the benchmark)
ENC = SentenceTransformer("BAAI/bge-m3", device="cuda"); ENC.max_seq_length = 256; ENC.half()
def enc(texts, bs=128):
    return ENC.encode([head_tail(t, 1200) for t in texts], batch_size=bs,
                      normalize_embeddings=True, show_progress_bar=False).astype(np.float32)
Er = enc(real.text_n.tolist())

X = np.vstack([Er, Es])
yl = np.r_[real.label.to_numpy(), syn.label.to_numpy()]
ya = np.r_[real.action.to_numpy(), syn.action.to_numpy()].astype(int)
yt = np.vstack([tech_matrix(real.techniques), tech_matrix(syn.techniques)])
sw = np.r_[np.full(len(real), 1.0), np.full(len(syn), 1.0)]      # synthetic weight 1.0 (best in the sweep)

mk = lambda: LogisticRegression(C=3, max_iter=3000, class_weight="balanced")
head_label  = mk().fit(X, yl, sample_weight=sw)
head_action = mk().fit(X, ya, sample_weight=sw)
head_tech = [mk().fit(X, yt[:, j], sample_weight=sw) if yt[:, j].min() != yt[:, j].max() else None for j in range(6)]
print("heads trained", X.shape)

synthetic rows kept: 5627 of 5638


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

heads trained (5713, 1024)


Cell 2: time 5,000 test rows first

In [20]:
te_full = pd.read_csv(os.path.join(COMP, "test.csv"))
te_full["text_n"] = te_full.text.map(robust)
t0 = time.time(); _ = enc(te_full.text_n.iloc[:5000].tolist(), bs=256); dt = time.time() - t0
print(round(dt, 1), "s per 5000 ->", round(dt * 100 / 60, 1), "min for 500k (estimate)")

9.0 s per 5000 -> 15.1 min for 500k (estimate)


*encode all 500k (resumable)*

In [21]:
os.makedirs("test_emb", exist_ok=True)
CH = 50_000
for s in range(0, len(te_full), CH):
    f = f"test_emb/bgem3_{s}.npy"
    if os.path.exists(f): continue
    t0 = time.time()
    v = enc(te_full.text_n.iloc[s:s + CH].tolist(), bs=256).astype(np.float16)
    np.save(f, v)
    print(s, "done", round(time.time() - t0), "s", flush=True)

0 done 61 s
50000 done 52 s
100000 done 52 s
150000 done 52 s
200000 done 52 s
250000 done 52 s
300000 done 52 s
350000 done 52 s
400000 done 52 s
450000 done 52 s


In [22]:
Et = np.vstack([np.load(f"test_emb/bgem3_{s}.npy") for s in range(0, len(te_full), CH)]).astype(np.float32)
assert len(Et) == len(te_full), (len(Et), len(te_full))

Pl = head_label.predict_proba(Et); Pa = head_action.predict_proba(Et)
Pt = np.stack([h.predict_proba(Et)[:, 1] if h is not None else np.zeros(len(Et)) for h in head_tech], axis=1)

lab = head_label.classes_[Pl.argmax(1)]
benign = np.isin(lab, (0, 5))

act = head_action.classes_[Pa.argmax(1)].copy()
cls_a = list(head_action.classes_)
nz = [i for i, c in enumerate(cls_a) if c != 0]
fix = ~benign & (act == 0)                                  # a scam always asks for something
act[fix] = np.array(cls_a)[nz][Pa[fix][:, nz].argmax(1)]
act[benign] = 0

T = Pt >= 0.5
T[benign] = False
none_scam = ~benign & ~T.any(1)                             # a scam uses at least one technique
T[none_scam, Pt[none_scam].argmax(1)] = True
tech_str = ["|".join(TECH[j] for j in np.flatnonzero(r)) or "NONE" for r in T]

conf = np.clip(Pl.max(1), 0.30, 0.85).round(4)              # capped: a leaderboard gap is likely, avoid overconfidence

sub = pd.DataFrame({"id": te_full.id, "label": lab, "techniques": tech_str, "action": act, "confidence": conf})
out = os.path.join(COMP, "submission_bgem3_synth.csv")
sub.to_csv(out, index=False)
print(sub.shape, "missing:", int(sub.isna().sum().sum()), "|", out)
print(sub.label.value_counts(normalize=True).sort_index().round(3).to_dict())
print(sub.head())

(500000, 5) missing: 0 | C:\ProjectHub\neurahack\nsucec-neurahack-2026-preliminary\submission_bgem3_synth.csv
{0: 0.263, 1: 0.221, 2: 0.119, 3: 0.09, 4: 0.13, 5: 0.177}
      id  label techniques  action  confidence
0  20001      3         T6       4      0.8500
1  20002      1      T2|T5       1      0.7241
2  20003      1   T1|T2|T5       1      0.8500
3  20004      0       NONE       0      0.5479
4  20005      1      T1|T2       1      0.8500


In [23]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
import random

yr = real.label.to_numpy(); ar = real.action.to_numpy().astype(int); tr_m = tech_matrix(real.techniques)
Gr = real.text.map(grp).factorize()[0]
rng = random.Random(1)
Er_n = enc([robust(corrupt(t, rng)) for t in real.text])       # corrupted copies -> robustness estimate
ys_l = syn.label.to_numpy(); ys_a = syn.action.to_numpy().astype(int); ys_t = tech_matrix(syn.techniques)

def macro(a, b):
    labs = np.union1d(np.unique(a), np.unique(b))
    return f1_score(a, b, labels=labs, average="macro", zero_division=0)

def fit_heads(X, yl, ya, yt):
    mk = lambda: LogisticRegression(C=3, max_iter=3000, class_weight="balanced")
    return (mk().fit(X, yl), mk().fit(X, ya),
            [mk().fit(X, yt[:, j]) if yt[:, j].min() != yt[:, j].max() else None for j in range(6)])

def predict(H, E):
    hl, ha, ht = H
    Pl, Pa = hl.predict_proba(E), ha.predict_proba(E)
    Pt = np.stack([h.predict_proba(E)[:, 1] if h is not None else np.zeros(len(E)) for h in ht], 1)
    lab = hl.classes_[Pl.argmax(1)]; ben = np.isin(lab, (0, 5))
    ca = np.array(ha.classes_); act = ca[Pa.argmax(1)].copy()
    nz = np.flatnonzero(ca != 0); fix = ~ben & (act == 0)
    if fix.any() and len(nz): act[fix] = ca[nz][Pa[fix][:, nz].argmax(1)]
    act[ben] = 0
    T = Pt >= 0.5; T[ben] = False
    ns = ~ben & ~T.any(1); T[ns, Pt[ns].argmax(1)] = True
    return lab, act, T, np.clip(Pl.max(1), 0.30, 0.85)

def tech_f1(Tt, Tp):
    fs = [f1_score(Tt[:, j], Tp[:, j], zero_division=0) for j in range(6) if Tt[:, j].any() or Tp[:, j].any()]
    return float(np.mean(fs))

def cv_all(use_syn=True, tau=0.65, seeds=range(3)):
    rows = []
    for s in seeds:
        for tr_i, va_i in StratifiedGroupKFold(5, shuffle=True, random_state=s).split(Er, yr, Gr):
            X, yl, ya, yt = Er[tr_i], yr[tr_i], ar[tr_i], tr_m[tr_i]
            if use_syn:
                k = (Es @ Er[va_i].T).max(1) < tau                  # drop synthetic rows close to held-out messages
                X = np.vstack([X, Es[k]]); yl = np.r_[yl, ys_l[k]]; ya = np.r_[ya, ys_a[k]]; yt = np.vstack([yt, ys_t[k]])
            H = fit_heads(X, yl, ya, yt)
            lab, act, T, conf = predict(H, Er[va_i])
            labn, *_ = predict(H, Er_n[va_i])
            A = macro(yr[va_i], lab); Rb = macro(yr[va_i], labn)
            B = tech_f1(tr_m[va_i], T); C = macro(ar[va_i], act)
            cal = 1 - np.mean((conf - (lab == yr[va_i])) ** 2)
            comp = (0.40*A + 0.15*B + 0.10*C + 0.10*Rb + 0.05*cal) / 0.80   # weights renormalised, hidden slices missing
            rows.append(dict(A=A, B=B, C=C, robust=Rb, calib=cal, composite=comp))
    return pd.DataFrame(rows).mean().round(3).to_dict()

print("real only        ", cv_all(use_syn=False))
print("with synthetic   ", cv_all(use_syn=True, tau=0.65))
print("with synthetic .6", cv_all(use_syn=True, tau=0.60))

real only         {'A': 0.822, 'B': 0.704, 'C': 0.612, 'robust': 0.789, 'calib': 0.65, 'composite': 0.759}
with synthetic    {'A': 0.932, 'B': 0.796, 'C': 0.879, 'robust': 0.882, 'calib': 0.929, 'composite': 0.893}
with synthetic .6 {'A': 0.922, 'B': 0.743, 'C': 0.839, 'robust': 0.888, 'calib': 0.895, 'composite': 0.872}


In [25]:
import os
print(os.getcwd())
print([f for f in os.listdir(".") if f.endswith(".ipynb")])


C:\Users\onimr
['24101121_427_lab2.ipynb', 'Copy of advancedBMI2.ipynb', 'Untitled.ipynb']
